In [45]:
import numpy as np
import pandas as pd

from sklearn.svm import SVR
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler

In [46]:
df_svr = pd.read_csv("complated_clean_data.csv")

In [47]:
df_svr.head(5)

,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,0.876693,1.989608,1.008484,0.925295,-0.038259,-1.447797,-1.108065,-0.948655,0.303668,35154898
1,1.238767,1.230524,1.008484,-0.704701,0.619638,-0.330404,0.902474,1.054124,-0.664835,26710893
2,-1.155940,-0.287644,0.106553,-0.355416,-1.354054,2.282091,-1.108065,1.054124,-0.093667,11216242
3,0.109526,0.471440,0.106553,0.284939,0.525653,-0.141549,-1.108065,-0.948655,-0.471135,21984310
4,0.064715,0.471440,1.008484,2.031364,0.995579,-1.825507,-1.108065,1.054124,-0.193000,25080429


In [48]:
x = df_svr.drop('house_price_inr',axis=1)
y = df_svr['house_price_inr']

In [49]:
from sklearn.model_selection import train_test_split,KFold,cross_val_score

In [50]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [51]:
len(x_train)

2871

In [52]:
len(x_test)

718

In [53]:
svr = TransformedTargetRegressor(
    
    regressor=SVR(
        kernel='linear',
        C=10,
        gamma='scale',
        epsilon=0.01
    ),
    
    transformer=StandardScaler()
)

In [54]:
svr.fit(x_train,y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","SVR(C=10, eps...rnel='linear')"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10


In [55]:
p_test = svr.predict(x_test) 

In [56]:
result = pd.DataFrame({
    'accual_values_price' : y_test.values,
    'predicted_house_price' : p_test
})

result

,accual_values_price,predicted_house_price
0,32691574,3.003625e+07
1,25525240,2.697668e+07
2,22740225,2.050385e+07
3,12718440,1.607270e+07
4,20657437,2.175081e+07
...,...,...
713,16954740,1.874442e+07
714,13520657,1.488949e+07
715,19403671,1.817469e+07
716,4225646,-1.418560e+05


In [57]:
score = r2_score(y_test,p_test) * 100
print(f"r2 score : {score}")

r2 score : 91.13013261642618


In [58]:
p_train = svr.predict(x_train) 

In [59]:
result = pd.DataFrame({
    'accual_values_price' : y_train.values,
    'predicted_house_price' : p_train
})

result

,accual_values_price,predicted_house_price
0,11996712,1.353255e+07
1,17020376,1.498245e+07
2,35531361,3.095777e+07
3,12257583,1.188066e+07
4,15698338,1.712131e+07
...,...,...
2866,26365052,2.634609e+07
2867,28146348,2.649297e+07
2868,34377748,3.315806e+07
2869,18941578,1.972384e+07


In [60]:
score_train = r2_score(y_train,p_train) * 100
print(f"r2 score : {score_train}")

r2 score : 91.35221448830578


In [61]:
mae = mean_absolute_error(y_test,p_test)
print(f"mean absolute error : {mae}")

mean absolute error : 1855717.4532679257


In [62]:
mse = mean_squared_error(y_test,p_test)
print(f"mean squared error : {mse}")

mean squared error : 5792911744612.267


In [63]:
kf = KFold(n_splits=5,random_state=42,shuffle=True)

In [42]:
cross_score = cross_val_score(svr,x,y,cv=kf,scoring='r2')

In [43]:
print(cross_score)

[0.90274228 0.90355713 0.90044412 0.90902023 0.89825663]


In [44]:
avg_score = cross_score.mean()
print(f"Avrage score : {avg_score}")

Avrage score : 0.9028040772332888


# Final Conclusion – Support Vector Regression (SVR)

| Metric | Value | Interpretation |
|----------|--------|----------------|
| Training R² Score | 91.35% | Model learned training data effectively |
| Testing R² Score | 91.13% | Strong prediction performance on unseen data |
| Average Cross Validation Score | 90.28% | Stable and reliable model performance |
| MAE | 1,855,717.45 | Average prediction error |
| MSE | 5,792,911,744,612.27 | Squared prediction error |
| Train-Test Difference | 0.22% | Extremely small difference indicates excellent generalization |
| Model Performance | Excellent |
| Overfitting Status | Very Low |

## Model Status

✅ Good Model  
❌ Worst Model: No

## Final Business Conclusion

The Support Vector Regression (SVR) model achieved a training score of **91.35%** and a testing score of **91.13%**. The average cross-validation score of **90.28%** indicates stable and consistent performance across multiple folds. The difference between training and testing scores is only **0.22%**, which indicates extremely low overfitting and excellent generalization capability.

The model demonstrates strong predictive performance with low prediction error and can effectively predict house prices on unseen data. Therefore, the Support Vector Regression (SVR) model can be considered a **good and reliable model for house price prediction**.